<a href="https://colab.research.google.com/github/AISHIK1/NN-with-Pytorch/blob/main/fanshion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import cv2
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection import train_test_split
import torch.nn as nn

In [2]:
import os

os.environ['KAGGLE_USERNAME'] = 'aishik996'
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_61236d6a08ac73ba31896583260ad688'



!kaggle datasets download -d zalando-research/fashionmnist

Dataset URL: https://www.kaggle.com/datasets/zalando-research/fashionmnist
License(s): other
100% 68.8M/68.8M [00:06<00:00, 11.5MB/s]



In [3]:
import zipfile

with zipfile.ZipFile('/content/fashionmnist.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [4]:
df_train=pd.read_csv('/content/fashion-mnist_train.csv')

In [5]:
df_test=pd.read_csv('/content/fashion-mnist_test.csv')

In [6]:
df_test.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,9,8,...,103,87,56,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,14,53,99,...,0,0,0,0,63,53,31,0,0,0
3,2,0,0,0,0,0,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df=pd.concat([df_train,df_test],axis=0)

In [8]:
df_train.shape

(60000, 785)

In [9]:
df_test.shape

(10000, 785)

In [10]:
df.shape

(70000, 785)

In [11]:
y=df.iloc[:,0].values

In [12]:
X=df.iloc[:,1:].values

In [13]:
#standerize
X=X/255.0

In [14]:
X

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.00392157,
        0.        ],
       [0.        , 0.00392157, 0.01176471, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [16]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sunig device : {device}")

Sunig device : cuda


In [17]:
#creating custom dataset

class mydataset(Dataset):

  def __init__(self,features,labels):

    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)


  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self,index):

    return self.features[index],self.labels[index]

In [18]:
train_dataset=mydataset(X_train,y_train)
test_dataset=mydataset(X_test,y_test)

In [19]:
train_loader=DataLoader(train_dataset,batch_size=128,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=128,shuffle=True,pin_memory=True)

In [57]:
from torch.nn.modules.activation import Softmax
#define the nn class

class Model(nn.Module):

  def __init__(self,num_features,num_labels):

    super().__init__()

    self.model=nn.Sequential(
        nn.Linear(num_features,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,num_labels),
        nn.Softmax()
        )


  def forward(self,features):

    out=self.model(features)

    return out

In [68]:
learning_rate=0.1
epochs=50

model=Model(X_train.shape[1],10)

model=model.to(device)

optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

loss_function=nn.CrossEntropyLoss()

In [69]:
for epoch in range(epochs):

  total_loss=0

  for batch_features,batch_label in train_loader:

    batch_features=batch_features.to(device)
    batch_label=batch_label.to(device)
    y_pred=model(batch_features)

    loss=loss_function(y_pred,batch_label)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss+=loss.item()

  avg_loss=total_loss/len(batch_features)
  print(f'epochs : {epoch+1}  avg_loss : {avg_loss}')

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


epochs : 1  avg_loss : 12.521158805117011
epochs : 2  avg_loss : 11.570758448913693
epochs : 3  avg_loss : 11.45338031463325
epochs : 4  avg_loss : 11.400318475440145
epochs : 5  avg_loss : 11.349950490519404
epochs : 6  avg_loss : 11.324876530095935
epochs : 7  avg_loss : 11.302089761942625
epochs : 8  avg_loss : 11.266241235658526
epochs : 9  avg_loss : 11.16142057068646
epochs : 10  avg_loss : 11.072960449382663
epochs : 11  avg_loss : 11.038654990494251
epochs : 12  avg_loss : 11.008638996630907
epochs : 13  avg_loss : 10.992042288184166
epochs : 14  avg_loss : 10.965486845001578
epochs : 15  avg_loss : 10.952755961567163
epochs : 16  avg_loss : 10.93098258599639
epochs : 17  avg_loss : 10.926416397094727
epochs : 18  avg_loss : 10.915191229432821
epochs : 19  avg_loss : 10.895208476111293
epochs : 20  avg_loss : 10.8912373483181
epochs : 21  avg_loss : 10.879578813910484
epochs : 22  avg_loss : 10.864462373778224
epochs : 23  avg_loss : 10.86086163111031
epochs : 24  avg_loss : 10

In [70]:
model.eval()

Model(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
    (9): Softmax(dim=None)
  )
)

In [71]:
total=0

correct=0

with torch.no_grad():

  for batch_feature,batch_label in train_loader:

    batch_feature=batch_feature.to(device)
    batch_label=batch_label.to(device)

    outputs=model(batch_feature)

    _,predicted=torch.max(outputs,1)

    total+=batch_label.shape[0]

    correct+=(predicted==batch_label).sum().item()

  accuracy=correct/total

  print(f'accuracy : {accuracy}')

accuracy : 0.9195


In [62]:
predicted

tensor([3, 7, 1, 9, 2, 0, 6, 3, 7, 8, 9, 3, 5, 9, 1, 4, 4, 0, 9, 8, 2, 9, 5, 9,
        3, 3, 1, 5, 4, 4, 2, 6, 6, 8, 6, 0, 9, 4, 2, 3, 8, 6, 6, 9, 9, 5, 2, 2,
        6, 8, 2, 8, 0, 4, 6, 5, 0, 9, 1, 6, 7, 3, 2, 7], device='cuda:0')

In [63]:
predicted[0]

tensor(3, device='cuda:0')

In [64]:
predicted

tensor([3, 7, 1, 9, 2, 0, 6, 3, 7, 8, 9, 3, 5, 9, 1, 4, 4, 0, 9, 8, 2, 9, 5, 9,
        3, 3, 1, 5, 4, 4, 2, 6, 6, 8, 6, 0, 9, 4, 2, 3, 8, 6, 6, 9, 9, 5, 2, 2,
        6, 8, 2, 8, 0, 4, 6, 5, 0, 9, 1, 6, 7, 3, 2, 7], device='cuda:0')

In [72]:
total=0

correct=0

with torch.no_grad():

  for batch_feature,batch_label in test_loader:

    batch_feature=batch_feature.to(device)
    batch_label=batch_label.to(device)

    outputs=model(batch_feature)

    _,predicted=torch.max(outputs,1)

    total+=batch_label.shape[0]

    correct+=(predicted==batch_label).sum().item()

  accuracy=correct/total

  print(f'accuracy : {accuracy}')

accuracy : 0.8892142857142857


In [66]:
device

device(type='cuda')